<a href="https://colab.research.google.com/github/laurencoetzee001/african_marriage_structures/blob/main/haiku_marriage_classifier_twostage_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marriage Practice Classifier — Two-Stage Pipeline
**Google Colab | Anthropic Claude Haiku | Codebook v6.0**

### Codebook v6 changes (03/04/2026)
Eight new variables added to Stage 2 to capture richer historical detail:

| # | Variable | Type | Notes |
|---|----------|------|-------|
| 13 | `13_wives_no` | integer or null | Number of wives stated in text; null if not specified |
| 13n | `13_wives_no_notes` | string or null | Additional context on wife limits (e.g. "up to four by Islamic law") |
| 14 | `14_concubines` | "yes" / "no" / null | Are concubines mentioned as permitted? Null = not listed |
| 14n | `14_concubines_notes` | string or null | Status, number, or circumstances of concubines |
| 15 | `15_dowry` | "yes" / "no" / null | Is a dowry or bride-wealth transaction mentioned? Null = not listed |
| 15n | `15_dowry_notes` | string or null | Price, goods, or conditions if given |
| 16 | `16_gender_roles` | string or null | Described only when wife's status is near-slave level; null otherwise |
| 17 | `17_legality` | string or null | Any stated law, chief's decree, religious rule, or ethnic norm governing marriage |
| 18 | `18_kingdoms` | string array or null | African kingdoms, chiefdoms, or royal figures named in text |
| 19 | `19_geo_features` | string array or null | Geographic features (rivers, mountains, lakes, ports) that locate the account |
| 20 | `20_historian_flag` | string or null | Brief note flagging rows of exceptional value for detailed case-study reading |

**`20_historian_flag` guidance**: populate ONLY when the text contains detailed, specific, and unusual information about African marriage structures that goes beyond a passing reference — e.g. a price list for bride-wealth, a described legal dispute, a first-hand account of a wedding ceremony, or an economically significant marriage alliance. The note should say WHY it is valuable (e.g. "detailed bride-wealth price list: cattle + cloth; named amounts").

### Codebook v5 changes (26/03/2026)
- **`3_pol_struc` simplified**: `seq_replace` (code 4) removed. Codes now 1–3 only. `varied` always flagged.
- **Exceptional behaviour flagging**: LLM must populate `11_conf_note` AND add `"EXCEPTIONAL"` to `12_validation_flags` for anomalous cases.
- **No more `unclear` or `unspecified` codes**: LLM commits to best-fit code + flags if <70% confident.

### Inherited from v3/v4
- `1_mar_prac` code 4 = `not_applicable`
- Harem → polygamy (code 2)
- `12_validation_flags`: per-variable flag when LLM confidence < 70%

### Pipeline
- **Stage 1** (cheap): Is this snippet relevant? NO → skip. YES → Stage 2.
- **Stage 2** (full codebook): Apply all variables. Flag uncertain codes.
- **Cell 8** (human review): Step through flagged rows interactively and reconcile.

### Before running
1. Click the **🔑 key icon** → add secret `ANTHROPIC_API_KEY` → toggle Notebook access ON
2. Update `INPUT_PATH` and `OUTPUT_DIR` in Cell 2
3. Run Cells 1–7, then Cell 8 for human review


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — INSTALL & MOUNT GOOGLE DRIVE
# Run this cell first, every session.
# ════════════════════════════════════════════════════════

!pip install -q anthropic

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

print()
print('✓ anthropic installed')
print('✓ Google Drive mounted at /content/drive/MyDrive')
print()
print('Next steps:')
print('  1. Set DRIVE_PROJECT_DIR in Cell 2 to your preferred Drive folder.')
print('  2. Upload your input .xlsx to that folder (or keep it in /content/).')
print('  3. Run Cells 2 → 7, then Cell 8 for human review.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.3/478.3 kB 8.7 MB/s eta 0:00:00
Mounted at /content/drive

✓ anthropic installed
✓ Google Drive mounted at /content/drive/MyDrive

Next steps:
  1. Set DRIVE_PROJECT_DIR in Cell 2 to your preferred Drive folder.
  2. Upload your input .xlsx to that folder (or keep it in /content/).
  3. Run Cells 2 → 7, then Cell 8 for human review.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ════════════════════════════════════════════════════════

from google.colab import userdata
from pathlib import Path
import os

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

# ── Google Drive project folder ───────────────────────────────────────────
# All outputs (CSVs, JSON results, row-by-row temp files) go here.
# Change this to match your own Drive structure.
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/TEACH_marriage_pipeline'

# ── Input file ────────────────────────────────────────────────────────────
# Option A — file lives on Drive (recommended: survives session resets)
INPUT_PATH = f'{DRIVE_PROJECT_DIR}/input/your_data.xlsx'

# Option B — file uploaded to Colab's ephemeral /content/ space
# INPUT_PATH = '/content/your_data.xlsx'

# ── Output directory (on Drive — auto-created if absent) ─────────────────
OUTPUT_DIR = f'{DRIVE_PROJECT_DIR}/output'

# ── Column and model settings ─────────────────────────────────────────────
TEXT_COLUMN   = 'full_text'
MAX_ROWS      = 100        # set to None for full dataset; 100 for a pilot run

MODEL         = 'claude-haiku-4-5-20251001'
TEMPERATURE   = 0.2
BATCH_SIZE    = 100        # print a progress summary every N rows
REQUEST_DELAY = 1.0        # seconds between API calls

# ── Drive auto-save settings ──────────────────────────────────────────────
# The ProgressTracker saves every individual row to Drive as it completes,
# so you can always resume from the last completed row after an interruption.
# DRIVE_SYNC_EVERY controls how often an additional explicit flush is triggered
# (on top of the per-row save) to ensure the checkpoint file is written.
DRIVE_SYNC_EVERY = 10      # flush Drive sync every N rows (set to 1 for maximum safety)

# ── Validation ────────────────────────────────────────────────────────────
assert ANTHROPIC_API_KEY, (
    'ANTHROPIC_API_KEY not found. '
    'Add it via the \U0001f511 Secrets panel (key icon) in the left sidebar.'
)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if not Path(INPUT_PATH).exists():
    print(f'⚠️  Input file not found at: {INPUT_PATH}')
    print('    Update INPUT_PATH above, or upload the file to Drive and try again.')
else:
    print(f'✓ Input file found : {INPUT_PATH}')

print(f'✓ API key loaded')
print(f'✓ Model          : {MODEL}')
print(f'✓ Output dir     : {OUTPUT_DIR}')
print(f'✓ Drive sync     : every row (temp files) + checkpoint every {DRIVE_SYNC_EVERY} rows')
print()
print('Drive save strategy:')
print('  • Every completed row → Drive temp file immediately (row_XXXXXX.json)')
print('  • Checkpoint file updated every row → resume-safe on interruption')
print(f'  • Explicit Drive flush every {DRIVE_SYNC_EVERY} rows via DRIVE_SYNC_EVERY')
print('  • Final JSON + CSVs written to Drive at end of Cell 6 and Cell 7')


⚠️  Input file not found at: /content/drive/MyDrive/TEACH_marriage_pipeline/input/your_data.xlsx
    Update INPUT_PATH above, or upload the file to Drive and try again.
✓ API key loaded
✓ Model          : claude-haiku-4-5-20251001
✓ Output dir     : /content/drive/MyDrive/TEACH_marriage_pipeline/output
✓ Drive sync     : every row (temp files) + checkpoint every 10 rows

Drive save strategy:
  • Every completed row → Drive temp file immediately (row_XXXXXX.json)
  • Checkpoint file updated every row → resume-safe on interruption
  • Explicit Drive flush every 10 rows via DRIVE_SYNC_EVERY
  • Final JSON + CSVs written to Drive at end of Cell 6 and Cell 7


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — EMBEDDED PROMPTS  (Codebook v6)
#
# KEY RULES (inherited): No 'unclear' or 'unspecified' codes.
# LLM always commits to best-fit code + flags if <70% confident.
# v6 adds: 13_wives_no/notes, 14_concubines/notes, 15_dowry/notes,
#          16_gender_roles, 17_legality, 18_kingdoms, 19_geo_features,
#          20_historian_flag
# ════════════════════════════════════════════════════════

STAGE_1_PROMPT = "You are a relevance filter for a historical research project on marriage practices in precolonial Africa (1500-1900), drawing on European travel accounts.\n\nAssess whether the text extract below is applicable for marriage structure coding.\n\nA snippet PASSES (applicable: true) only if ALL three criteria are met:\n1. HUMAN SUBJECTS \u2014 the text describes human beings, not animals or objects\n2. READABLE \u2014 the text is coherent and interpretable; not so corrupted by OCR errors that it cannot be understood\n3. MARRIAGE CONTENT \u2014 the text contains at least one reference to: marriage, wives, husbands, marital customs, polygamy, polygyny, polyandry, monogamy, harem, harems, bride-price, betrothal, or related social structures\n\nA snippet FAILS (applicable: false) if ANY of the following apply:\n- No marriage-related content whatsoever\n- Text is a contents page, chapter list, or table of contents (even if it mentions relevant chapter titles)\n- Text consists primarily of page numbers, headings, or navigation entries\n- Text is so corrupted by OCR errors, garbling, or truncation that its meaning cannot be determined\n- Text discusses only trade, geography, flora, fauna, warfare, or other topics with no marriage reference\n- Text discusses animals or non-human subjects only\n\nWhen in doubt, PASS the row \u2014 Stage 2 will handle borderline cases with 1_mar_prac = 4 (not_applicable).\n\nReturn ONLY valid JSON. No explanation, no markdown, no text before or after:\n{\"applicable\": true}\nOR\n{\"applicable\": false, \"reason\": \"<one short phrase>\"}"


STAGE_2_PROMPT = "# MARRIAGE PRACTICES CODING INSTRUMENT v6.0\n# Time Traveller Corpus | Precolonial Africa 1500-1900\n\n## TASK\nClassify the marriage structure described in the text extract.\n- Use ONLY information present in the text. Do NOT apply external cultural or historical knowledge.\n- Apply codes based on what the author reports, not what you infer about the society.\n- String fields must be copied VERBATIM \u2014 preserve original spelling, transliteration, and capitalisation exactly.\n- Where a conditional rule requires null, output null. Otherwise always output a code.\n\n## CORE RULE \u2014 NO UNCERTAIN CODES\nThere are no 'unclear' or 'unspecified' codes in this instrument.\nWhen you are uncertain, you MUST still select the single best-fit code based on available evidence,\nAND add that variable name to 12_validation_flags to request human review.\nDo NOT leave coded variables blank. Do NOT invent new codes.\n\n---\n\n## VARIABLE 0 \u2014 0_subject: Who Are the Individuals in This Marriage?\nNull if 1_mar_prac = 4.\nUse ONLY explicit textual evidence to assign. When the text does not name the parties,\nselect the most plausible code given the context AND add \"0_subject\" to 12_validation_flags.\n\n| Code | Label | Apply when |\n|------|-------|------------|\n| 1 | african_african | Both parties identified as African, same ethnic group or location |\n| 2 | african_african_diff | Both parties African but different ethnic groups or geographic locations |\n| 3 | african_foreign | One party African, one party European or foreign traveller |\n| 4 | foreign_foreign | Both parties European or foreign |\n\nWhen in doubt: if the extract is about an African ethnic group's customs \u2192 code 1.\nIf the author describes their own marriage to a local \u2192 code 3.\nAlways flag in 12_validation_flags if inferred rather than explicit.\n\n---\n\n## VARIABLE 1 \u2014 1_mar_prac: Marriage Practice Type\n\n| Code | Label | Apply when |\n|------|-------|------------|\n| 1 | monogamy_only | ONLY monogamous structure \u2014 one spouse per partner (any gender) |\n| 2 | polygamy_only | ONLY polygamous structure \u2014 multiple simultaneous spouses (includes polygyny AND polyandry) |\n| 3 | both | Two DISTINCT marriage practices described for DIFFERENT groups or societies within the same extract |\n| 4 | not_applicable | Contents page, table of contents, OCR too corrupted to read, error, or no marriage content \u2014 ALL variables null |\n\nSTOP CONDITION: IF 1_mar_prac = 4 \u2192 set ALL variables (including 0_subject) to null and stop.\n\nMonogamy (code 1): \"monogamy\"/\"monogamous\"/\"one wife\"/\"one husband\"/\"single spouse\" / legal prohibition of multiple spouses with no countervailing practice.\nPolygamy explicit (code 2): \"polygamy\"/\"polygyny\"/\"polyandry\"/\"multiple wives\"/\"several wives\"/\"two wives\"/\"many wives\"/\"plural wives\"/\"multiple husbands\"\nPolygamy implicit (code 2): \"his wives\" (plural) / ranked wives (\"first wife\"/\"senior wife\"/\"chief wife\") / \"co-wives\" / HAREM or HAREMS mentioned\nCode 3 (both): Two distinct groups each described with DIFFERENT structures. NOT for one group with both practices.\nCRITICAL: singular \"wife\" with no structure indicated \u2192 pick best-fit code (usually 1) + flag. Harem = code 2.\n\n---\n\n## VARIABLE 2 \u2014 2_mar_diff: Prevalence of the Practice\nNull if 1_mar_prac = 4.\n\n| Code | Label | Apply when |\n|------|-------|------------|\n| 1 | widespread | General, common, normal, or universal in the population |\n| 2 | rare_exception | Exceptional, rare, or limited to specific individuals |\n\nWhen in doubt: custom or norm of a group \u2192 code 1. Limited to few individuals \u2192 code 2. Always flag.\n\n---\n\n## VARIABLE 3 \u2014 3_pol_struc: Polygamy Structure\nNull UNLESS 1_mar_prac = 2 or 3. When 1_mar_prac = 3: code the POLYGAMOUS union only.\n\n| Code | Label | Apply when |\n|------|-------|------------|\n| 1 | fixed_spouses | Normalised fixed number described (\"each man takes two wives\") |\n| 2 | varied | Variable number with implied limits (\"richer man, more wives\") \u2014 always flag |\n| 3 | unlimited | No restrictions on number stated or implied |\n\nWhen in doubt and text mentions harem or polygamy with no numeric detail \u2192 code 3 and flag.\n\n---\n\n## VARIABLE 4 \u2014 4_ethnic_group: Ethnic Group(s) Named\nSTRING ARRAY \u2014 verbatim from text, no normalisation. Null if none named.\nPreserve archaic transliterations exactly (e.g. \"Bechuanas\", \"Wanyamwesi\", \"Kafirs\").\nNull if 1_mar_prac = 4.\n\n---\n\n## VARIABLE 5 \u2014 5_soc_stat: Social Status of Practitioners\nNull if 1_mar_prac = 4.\n\n| Code | Label | Apply when |\n|------|-------|------------|\n| 1 | general_pop | Applies throughout the general population |\n| 2 | elite_wealthy | Explicitly limited to elite, wealthy, noble, or chiefly individuals |\n| 3 | eth_restric | Limited to a specific ethnic subgroup within a larger population |\n\nWhen in doubt: group custom without qualification \u2192 code 1. Wealth or rank as condition \u2192 code 2. Always flag.\n\n---\n\n## VARIABLE 6 \u2014 6_obs_type: Type of Observation\nNull if 1_mar_prac = 4.\n\n| Code | Label | Apply when |\n|------|-------|------------|\n| 1 | auth_direct | Author explicitly states direct personal witnessing |\n| 2 | auth_second_res | Named or described long-term resident of the area |\n| 3 | auth_hearsay | Unnamed informant / \"it is said\" / \"they say\" / \"I was told\" |\n| 4 | auth_gen | General statement with no attribution to a person or source |\n\n---\n\n## VARIABLE 7 \u2014 7_obs_location: Geographic Location Present\nNull if 1_mar_prac = 4.\n\n| Code | Apply when |\n|------|------------|\n| 1 | A named geographic location or place identifier is present in the extract |\n| 2 | No geographic location is present in the extract |\n\n---\n\n## VARIABLE 8 \u2014 8_cited_loc: Location Name(s)\nSTRING ARRAY \u2014 verbatim from text. Null if 7_obs_location != 1.\nPreserve original spellings (e.g. \"Timbuctoo\", \"the Zambeze\", \"Dahomey\"). List ALL named places.\n\n---\n\n## VARIABLE 9 \u2014 9_moral_op: Moral Opinion Present\nNull if 1_mar_prac = 4.\n\n| Code | Apply when |\n|------|------------|\n| 1 | moral_judge | Explicit moral, religious, or evaluative judgement about the practice |\n| 2 | no_judge | No moral judgement present |\n\nIndicators for code 1: (\"barbarous\"/\"degrading\"/\"sinful\"/\"uncivilised\"/\"revolting\"/\"commendable\") /\nreligious framing / comparative moral claim / desire for reform.\n\n---\n\n## VARIABLE 10 \u2014 10_stance: Verbatim Moral Remark\nSTRING \u2014 verbatim from text. Null if 9_moral_op != 1.\n\n---\n\n## VARIABLE 11 \u2014 11_code_conf: Overall Coding Confidence\nAlways required. Null only if 1_mar_prac = 4.\n\n| Code | Apply when |\n|------|------------|\n| 1 | high | >=95% confidence in accuracy of all coded variables |\n| 2 | medium | 60-94% \u2014 one or more variables uncertain |\n| 3 | low | <60% \u2014 substantial ambiguity throughout |\n\n## 11_conf_note\nString (<=30 words). Null only if 1_mar_prac = 4.\nALWAYS populate when ANY of the following apply: polyandry present; behaviour exceptional or anomalous;\nmixed-race or cross-ethnic marriage; marriage involving religious figures, enslaved persons, or captives.\nWrite a brief factual note, e.g. \"polyandry: one wife shared by brothers\".\n\n---\n\n## VARIABLE 12 \u2014 12_validation_flags: Per-Variable Human Validation Flags\nNull if 1_mar_prac = 4. Otherwise always required \u2014 return [] if no flags.\nAdd a variable name when: <70% confident in that variable's code; inferred not explicit; EXCEPTIONAL case.\nAdd \"EXCEPTIONAL\" as a string entry for: polyandry, cross-ethnic marriage, religious figures, captives.\n\n---\n\n## VARIABLE 13 \u2014 13_wives_no: Number of Wives Stated\nINTEGER or null. Null if 1_mar_prac = 4.\nRecord the stated maximum, minimum, or typical number as an integer.\nIf a range is given (\"two or three\"), record the UPPER bound.\nIf only qualitative (\"many wives\", \"a great number\") with no number, record null.\n\n## 13_wives_no_notes\nSTRING (<=40 words) or null.\nAdditional context on wife-number limits (e.g. \"maximum of four by Islamic law; in practice most take one or two\").\nMay be populated even when 13_wives_no is null (e.g. \"as many as he can afford\").\n\n---\n\n## VARIABLE 14 \u2014 14_concubines: Concubines Mentioned\nSTRING: \"yes\", \"no\", or null. Null if 1_mar_prac = 4.\n- \"yes\": concubines explicitly mentioned as permitted or present\n- \"no\": text explicitly states concubines are NOT permitted\n- null: concubines not mentioned at all (do not infer from absence)\n\n## 14_concubines_notes\nSTRING (<=40 words) or null. Populate when 14_concubines = \"yes\" with additional detail\n(legal status of children, upper limit, ethnic or religious source of rule).\n\n---\n\n## VARIABLE 15 \u2014 15_dowry: Dowry or Bride-Wealth Mentioned\nSTRING: \"yes\", \"no\", or null. Null if 1_mar_prac = 4.\n- \"yes\": a dowry, bride-price, bride-wealth, or payment for a wife is explicitly mentioned\n- \"no\": text explicitly states no such payment exists\n- null: topic not mentioned (do not infer from absence)\n\n## 15_dowry_notes\nSTRING (<=40 words) or null. Populate when 15_dowry = \"yes\" and price, goods, or conditions given\n(e.g. \"ten cattle and a bolt of cloth\", \"negotiated between families\", \"paid to father of bride\").\n\n---\n\n## VARIABLE 16 \u2014 16_gender_roles: Extreme Subservience of Wife\nSTRING (<=60 words) or null. Null if 1_mar_prac = 4.\nONLY populate when the text EXPLICITLY describes a wife's status as near-slave level:\nconfined, not allowed to leave, compared to property, described as bought or owned, entirely without autonomy.\nIf the description is of ordinary gender difference or mild inequality, leave null.\nQuote or closely paraphrase the relevant passage verbatim.\n\n---\n\n## VARIABLE 17 \u2014 17_legality: Stated Laws or Rules Governing Marriage\nSTRING (<=60 words) or null. Null if 1_mar_prac = 4.\nONLY populate when the text explicitly states a law, a chief's decree, a religious rule, or a named\nethnic norm governing marriage, wife number, divorce, or spousal dynamics.\nParaphrase concisely (e.g. \"Islamic law permits up to four wives\").\nDo NOT populate for informal customs described without legal framing.\n\n---\n\n## VARIABLE 18 \u2014 18_kingdoms: African Kingdoms, Chiefdoms, or Royal References\nSTRING ARRAY \u2014 verbatim from text, or null. Null if 1_mar_prac = 4.\nList any named African kingdoms, chiefdoms, sultanates, or named rulers/royals mentioned.\nPreserve archaic spellings exactly (e.g. \"Dahomey\", \"the Sultan of Zanzibar\", \"Mutesa's court\").\nNull if none named.\n\n---\n\n## VARIABLE 19 \u2014 19_geo_features: Geographic Features\nSTRING ARRAY \u2014 verbatim from text, or null. Null if 1_mar_prac = 4.\nList geographic features only: rivers, lakes, mountains, ports, valleys, coasts, islands.\nInclude the feature name as it appears (e.g. \"the Limpopo\", \"Lake Nyassa\", \"the Bight of Benin\").\nNOTE: Named settlements go in 8_cited_loc, not here. Only physical landscape features.\nNull if none mentioned.\n\n---\n\n## VARIABLE 20 \u2014 20_historian_flag: Historian/Economist Case-Study Flag\nSTRING (<=50 words) or null. Null if 1_mar_prac = 4.\nFlag ONLY when the text contains unusually rich, specific detail warranting close human reading:\n- Detailed bride-wealth price list (specific goods, animals, monetary amounts named)\n- First-hand account of a marriage ceremony or negotiation with procedural detail\n- Explicitly described legal dispute or court ruling about marriage\n- Named economic or political marriage alliance between kingdoms or chiefs\n- Unusually detailed harem organisation, hierarchy, or daily life\n- Quantitative data on marriage (number of wives per man across a community)\nWrite a short note explaining WHY flagged.\nDo NOT flag merely because polygamy is mentioned or a moral judgment is present.\n\n---\n\n## LOGICAL CONSTRAINTS\n\n| Condition | Required |\n|-----------|----------|\n| 1_mar_prac = 4 | ALL variables (incl. 0_subject) = null |\n| 1_mar_prac = 1 | 3_pol_struc = null |\n| 1_mar_prac = 2 or 3 | 3_pol_struc must have a value 1-3 |\n| 1_mar_prac = 3 | 3_pol_struc refers to polygamous union ONLY |\n| 7_obs_location != 1 | 8_cited_loc = null |\n| 7_obs_location = 1 | 8_cited_loc must be a non-empty string array |\n| 9_moral_op != 1 | 10_stance = null |\n| 9_moral_op = 1 | 10_stance must be a non-empty string |\n| 13_wives_no = null AND 13_wives_no_notes = null | Both null when no wife-number detail present |\n| 14_concubines = null | 14_concubines_notes must also be null |\n| 15_dowry = null | 15_dowry_notes must also be null |\n\n---\n\n## EXAMPLES\n\nText: \"The Waswahili, like all Mahommedans, are allowed by their law to take four wives; but the rich alone can afford to maintain more than one or two. I found this custom degrading to the women.\"\nOutput: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 1, \"3_pol_struc\": 1, \"4_ethnic_group\": [\"Waswahili\"], \"5_soc_stat\": 1, \"6_obs_type\": 4, \"7_obs_location\": 2, \"8_cited_loc\": null, \"9_moral_op\": 1, \"10_stance\": \"I found this custom degrading to the women.\", \"11_code_conf\": 2, \"11_conf_note\": \"5_soc_stat: law universal but practice wealth-dependent\", \"12_validation_flags\": [\"5_soc_stat\"], \"13_wives_no\": 4, \"13_wives_no_notes\": \"Islamic law permits four; in practice wealth limits most to one or two\", \"14_concubines\": null, \"14_concubines_notes\": null, \"15_dowry\": null, \"15_dowry_notes\": null, \"16_gender_roles\": null, \"17_legality\": \"Mahommedan law permits up to four wives\", \"18_kingdoms\": null, \"19_geo_features\": null, \"20_historian_flag\": null}\n\nText: \"The Turks of this province maintain large harems, whilst the Christian Greeks of the same district take but a single wife.\"\nOutput: {\"0_subject\": 4, \"1_mar_prac\": 3, \"2_mar_diff\": 1, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Turks\", \"Greeks\"], \"5_soc_stat\": 1, \"6_obs_type\": 4, \"7_obs_location\": 2, \"8_cited_loc\": null, \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 2, \"11_conf_note\": \"0_subject ambiguous: Turks/Greeks may be local or foreign\", \"12_validation_flags\": [\"0_subject\"], \"13_wives_no\": null, \"13_wives_no_notes\": null, \"14_concubines\": null, \"14_concubines_notes\": null, \"15_dowry\": null, \"15_dowry_notes\": null, \"16_gender_roles\": null, \"17_legality\": null, \"18_kingdoms\": null, \"19_geo_features\": null, \"20_historian_flag\": null}\n\nText: \"The king of Dahomey, in his palace at Abomey, maintains some three thousand women, divided into wives of various ranks and concubines who serve as soldiers. Each wife received upon entry a payment of ten cowrie strings from the royal treasury, and those of the first rank commanded their own household. A woman taken into the palace could not leave it, nor could she refuse.\"\nOutput: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 2, \"3_pol_struc\": 3, \"4_ethnic_group\": null, \"5_soc_stat\": 2, \"6_obs_type\": 4, \"7_obs_location\": 1, \"8_cited_loc\": [\"Abomey\"], \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 1, \"11_conf_note\": \"royal harem ~3000; concubines serve as palace soldiers\", \"12_validation_flags\": [\"EXCEPTIONAL\"], \"13_wives_no\": null, \"13_wives_no_notes\": \"approximately three thousand women in the royal palace; ranked hierarchy\", \"14_concubines\": \"yes\", \"14_concubines_notes\": \"concubines serve as palace soldiers; distinct from ranked wives\", \"15_dowry\": \"yes\", \"15_dowry_notes\": \"ten cowrie strings paid to each wife upon entry from royal treasury\", \"16_gender_roles\": \"A woman taken into the palace could not leave it, nor could she refuse\", \"17_legality\": \"internal palace hierarchy: first-rank wives command their own households\", \"18_kingdoms\": [\"Dahomey\", \"king of Dahomey\"], \"19_geo_features\": null, \"20_historian_flag\": \"royal harem ~3000; concubines as soldiers; bride-payment in cowrie strings; full confinement. Strong case study for political economy of elite marriage.\"}\n\nText: \"It is said that in the region between the Lualaba and the Lomami rivers, the Basonge practice a most revolting form of polygamy, in which a man may accumulate wives without limit.\"\nOutput: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 1, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Basonge\"], \"5_soc_stat\": 1, \"6_obs_type\": 3, \"7_obs_location\": 1, \"8_cited_loc\": [\"Lualaba\", \"Lomami\"], \"9_moral_op\": 1, \"10_stance\": \"a most revolting form of polygamy\", \"11_code_conf\": 1, \"11_conf_note\": null, \"12_validation_flags\": [], \"13_wives_no\": null, \"13_wives_no_notes\": null, \"14_concubines\": null, \"14_concubines_notes\": null, \"15_dowry\": null, \"15_dowry_notes\": null, \"16_gender_roles\": null, \"17_legality\": null, \"18_kingdoms\": null, \"19_geo_features\": [\"Lualaba\", \"Lomami\"], \"20_historian_flag\": null}\n\nText: \"Among the Nairs of the Malabar coast, one woman is shared as wife by several brothers simultaneously.\"\nOutput: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 1, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Nairs\"], \"5_soc_stat\": 1, \"6_obs_type\": 4, \"7_obs_location\": 1, \"8_cited_loc\": [\"Malabar\"], \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 1, \"11_conf_note\": \"polyandry: one woman shared as wife by several brothers simultaneously\", \"12_validation_flags\": [\"EXCEPTIONAL\"], \"13_wives_no\": null, \"13_wives_no_notes\": null, \"14_concubines\": null, \"14_concubines_notes\": null, \"15_dowry\": null, \"15_dowry_notes\": null, \"16_gender_roles\": null, \"17_legality\": null, \"18_kingdoms\": null, \"19_geo_features\": [\"Malabar coast\"], \"20_historian_flag\": null}\n\nText: \"CHAPTER IV. Of the customs of the Kafirs \u2014 p. 47. Of the trade in ivory \u2014 p. 63.\"\nOutput: {\"0_subject\": null, \"1_mar_prac\": 4, \"2_mar_diff\": null, \"3_pol_struc\": null, \"4_ethnic_group\": null, \"5_soc_stat\": null, \"6_obs_type\": null, \"7_obs_location\": null, \"8_cited_loc\": null, \"9_moral_op\": null, \"10_stance\": null, \"11_code_conf\": null, \"11_conf_note\": null, \"12_validation_flags\": null, \"13_wives_no\": null, \"13_wives_no_notes\": null, \"14_concubines\": null, \"14_concubines_notes\": null, \"15_dowry\": null, \"15_dowry_notes\": null, \"16_gender_roles\": null, \"17_legality\": null, \"18_kingdoms\": null, \"19_geo_features\": null, \"20_historian_flag\": null}\n\n---\n\n## OUTPUT FORMAT\nReturn ONLY a single-line valid JSON object. No markdown, no explanation, no text before or after.\n{\"0_subject\": X, \"1_mar_prac\": X, \"2_mar_diff\": X, \"3_pol_struc\": X, \"4_ethnic_group\": X, \"5_soc_stat\": X, \"6_obs_type\": X, \"7_obs_location\": X, \"8_cited_loc\": X, \"9_moral_op\": X, \"10_stance\": X, \"11_code_conf\": X, \"11_conf_note\": X, \"12_validation_flags\": X, \"13_wives_no\": X, \"13_wives_no_notes\": X, \"14_concubines\": X, \"14_concubines_notes\": X, \"15_dowry\": X, \"15_dowry_notes\": X, \"16_gender_roles\": X, \"17_legality\": X, \"18_kingdoms\": X, \"19_geo_features\": X, \"20_historian_flag\": X}"


print(f'\u2713 Stage 1 prompt : {len(STAGE_1_PROMPT):,} characters')
print(f'\u2713 Stage 2 prompt : {len(STAGE_2_PROMPT):,} characters')
print()
print('v6 additions: 13_wives_no, 14_concubines, 15_dowry, 16_gender_roles,')
print('              17_legality, 18_kingdoms, 19_geo_features, 20_historian_flag')


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — IMPORTS, CLASSIFIER & PROGRESS TRACKER
# ════════════════════════════════════════════════════════

import json
import re
import time
import pandas as pd
import anthropic
from pathlib import Path
from datetime import datetime


class ProgressTracker:
    """Saves each row to Drive immediately. Resume-safe on interruption."""

    def __init__(self, output_dir: Path, document_name: str,
                 drive_sync_every: int = 10):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.document_name = document_name
        self.temp_dir = self.output_dir / f'temp_{document_name}'
        self.temp_dir.mkdir(exist_ok=True)
        self.checkpoint_file = self.temp_dir / 'checkpoint.json'
        self.drive_sync_every = drive_sync_every
        self.progress = self._load_checkpoint()

    def _load_checkpoint(self) -> dict:
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file) as f:
                return json.load(f)
        return {'completed_rows': [], 'start_time': datetime.now().isoformat()}

    def _save_checkpoint(self):
        self.progress['last_update'] = datetime.now().isoformat()
        with open(self.checkpoint_file, 'w') as f:
            json.dump(self.progress, f, indent=2)

    def should_process(self, row_number: int) -> bool:
        return row_number not in self.progress['completed_rows']

    def save_row(self, row_number: int, result: dict):
        # Write individual row result immediately to Drive
        path = self.temp_dir / f'row_{row_number:06d}.json'
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(
                {'row_number': row_number, 'result': result,
                 'processed_at': datetime.now().isoformat()},
                f, indent=2, ensure_ascii=False
            )
        if row_number not in self.progress['completed_rows']:
            self.progress['completed_rows'].append(row_number)
        self._save_checkpoint()

        # Periodic explicit Drive flush (belt-and-suspenders on top of per-row save)
        n_done = len(self.progress['completed_rows'])
        if self.drive_sync_every and n_done % self.drive_sync_every == 0:
            self._drive_flush()

    def _drive_flush(self):
        """Force Drive to sync by rewriting the checkpoint file with a flush marker."""
        try:
            self.progress['last_drive_flush'] = datetime.now().isoformat()
            self._save_checkpoint()
            # Colab flushes kernel writes to Drive on file-close, which happens above.
            # We also write a small sentinel file to confirm sync is current.
            sentinel = self.temp_dir / 'drive_sync.txt'
            sentinel.write_text(
                f'Last flush: {self.progress["last_drive_flush"]}\n'
                f'Rows completed: {len(self.progress["completed_rows"])}\n'
            )
        except Exception as e:
            print(f'   ⚠️  Drive flush warning (non-fatal): {e}')

    def load_all_results(self) -> list:
        results = []
        for path in sorted(self.temp_dir.glob('row_*.json')):
            with open(path, encoding='utf-8') as f:
                results.append(json.load(f)['result'])
        return results

    def finalize(self, all_results: list) -> Path:
        output_path = self.output_dir / f'{self.document_name}.json'
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(
                {'model': MODEL, 'codebook_version': 'v6.0',
                 'document_name': self.document_name,
                 'total_rows': len(all_results),
                 'processed_at': datetime.now().isoformat(),
                 'results': all_results},
                f, indent=2, ensure_ascii=False
            )
        print(f'\n✅ Results saved to Drive: {output_path}')
        return output_path


class TwoStageClassifier:
    """
    Two-stage pipeline — Codebook v6.

    Key v4 changes:
      - No unclear/unspecified codes. LLM always picks best-fit + flags.
      - 0_subject: codes 1-4 only (no code 5)
      - 2_mar_diff: codes 1-2 only (no code 3)
      - 3_pol_struc: codes 1-4 only (no code 5)
      - 5_soc_stat: codes 1-3 only (no code 4)
      - 6_obs_type: codes 1-4 only (no code 5)
      - 7_obs_location: codes 1-2 only (no code 3)
      - 9_moral_op: codes 1-2 only (no code 3)
      - max_tokens increased to 650 to handle fuller 12_validation_flags arrays
    """

    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.stats = {
            'total_rows': 0, 'filtered': 0, 'classified': 0,
            'not_applicable': 0, 'flagged': 0, 'errors': 0,
            'stage1_in': 0, 'stage1_out': 0, 'stage1_cache': 0,
            'stage2_in': 0, 'stage2_out': 0, 'stage2_cache': 0,
        }

    def _call(self, system_prompt: str, text: str, max_tokens: int,
              stat_prefix: str, max_retries: int = 3) -> dict:
        for attempt in range(max_retries):
            try:
                response = self.client.messages.create(
                    model=MODEL,
                    max_tokens=max_tokens,
                    temperature=TEMPERATURE,
                    system=[{'type': 'text', 'text': system_prompt,
                             'cache_control': {'type': 'ephemeral'}}],
                    messages=[{'role': 'user', 'content': text}]
                )
                u = response.usage
                self.stats[f'{stat_prefix}_in']    += getattr(u, 'input_tokens', 0)
                self.stats[f'{stat_prefix}_out']   += getattr(u, 'output_tokens', 0)
                self.stats[f'{stat_prefix}_cache'] += getattr(u, 'cache_read_input_tokens', 0)
                return {'text': response.content[0].text.strip()}
            except anthropic.RateLimitError:
                if attempt < max_retries - 1:
                    wait = 60 * (2 ** attempt)
                    print(f'   ⚠️  Rate limit — waiting {wait}s…')
                    time.sleep(wait)
                else:
                    return {'error': 'Rate limit exceeded'}
            except Exception as e:
                if attempt < max_retries - 1:
                    wait = 10 * (2 ** attempt)
                    print(f'   ⚠️  Retry {attempt+1} in {wait}s: {e}')
                    time.sleep(wait)
                else:
                    return {'error': str(e)}

    def _parse_json(self, raw: str) -> dict:
        m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
        json_str = m.group(1) if m else re.search(r'\{.*\}', raw, re.DOTALL).group(0)
        return json.loads(json_str)

    def process_row(self, text: str, row_number: int) -> dict:
        self.stats['total_rows'] += 1

        r1 = self._call(STAGE_1_PROMPT, text, 60, 'stage1')
        if 'error' in r1:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error', 'error': r1['error']}

        try:
            f = self._parse_json(r1['text'])
        except Exception:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error',
                    'error': f'Stage 1 JSON parse failed: {r1["text"][:80]}'}

        if not f.get('applicable', False):
            self.stats['filtered'] += 1
            return {'row_number': row_number, 'stage': 'filtered',
                    'reason': f.get('reason', 'not applicable')}

        r2 = self._call(STAGE_2_PROMPT, text, 1100, 'stage2')
        if 'error' in r2:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error', 'error': r2['error']}

        try:
            c = self._parse_json(r2['text'])
        except Exception:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error',
                    'error': f'Stage 2 JSON parse failed: {r2["text"][:80]}'}

        mp    = c.get('1_mar_prac')
        flags = c.get('12_validation_flags') or []

        if mp == 4:
            self.stats['not_applicable'] += 1
            stage = 'not_applicable'
        else:
            self.stats['classified'] += 1
            stage = 'classified'

        if isinstance(flags, list) and len(flags) > 0:
            self.stats['flagged'] += 1

        return {'row_number': row_number, 'stage': stage, 'classification': c}

    def print_stats(self):
        s = self.stats
        n = s['total_rows'] or 1
        for stage, label in [('stage1', 'Stage 1'), ('stage2', 'Stage 2')]:
            total = s[f'{stage}_in'] + s[f'{stage}_cache']
            hit   = (s[f'{stage}_cache'] / total * 100) if total else 0
            s[f'{stage}_hit'] = hit
        print(f'\n{"="*58}')
        print(f'PIPELINE STATS  ({MODEL}  |  Codebook v6)')
        print(f'{"="*58}')
        print(f'Total rows processed    : {s["total_rows"]:,}')
        print(f'Stage 1 filtered        : {s["filtered"]:,}  ({s["filtered"]/n*100:.1f}%)')
        print(f'Stage 2 classified      : {s["classified"]:,}')
        print(f'Stage 2 not_applicable  : {s["not_applicable"]:,}  (1_mar_prac=4)')
        print(f'Rows flagged for review : {s["flagged"]:,}  (12_validation_flags non-empty)')
        print(f'Errors                  : {s["errors"]:,}')
        print(f'---')
        print(f'Stage 1 tokens in/out/cached : {s["stage1_in"]:,} / {s["stage1_out"]:,} / {s["stage1_cache"]:,}  (hit: {s["stage1_hit"]:.1f}%)')
        print(f'Stage 2 tokens in/out/cached : {s["stage2_in"]:,} / {s["stage2_out"]:,} / {s["stage2_cache"]:,}  (hit: {s["stage2_hit"]:.1f}%)')
        print(f'{"="*58}')


print('✓ ProgressTracker and TwoStageClassifier (v6) defined')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — LOAD DATA
# ════════════════════════════════════════════════════════

df = pd.read_excel(INPUT_PATH)
if MAX_ROWS:
    df = df.head(MAX_ROWS)

assert TEXT_COLUMN in df.columns, (
    f"Column '{TEXT_COLUMN}' not found. Available: {list(df.columns)}"
)

print(f'✓ Data loaded: {len(df)} rows')
print(f'  Columns: {list(df.columns)}')
print(f'\nFirst text preview ({len(str(df.iloc[0][TEXT_COLUMN]))} chars):')
print(str(df.iloc[0][TEXT_COLUMN])[:400], '...')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — RUN TWO-STAGE PIPELINE
# ════════════════════════════════════════════════════════

print('='*60)
print('MARRIAGE PRACTICE CLASSIFIER — TWO-STAGE PIPELINE  v6.0')
print('='*60)
print(f'  Model: {MODEL}  |  Codebook: v6.0  |  Rows: {len(df)}')

classifier = TwoStageClassifier(ANTHROPIC_API_KEY)
tracker    = ProgressTracker(Path(OUTPUT_DIR), 'haiku_classifications_v6',
                             drive_sync_every=DRIVE_SYNC_EVERY)

completed = len(tracker.progress['completed_rows'])
if completed > 0:
    print(f'\n   ℹ️  Resuming: {completed}/{len(df)} rows already processed')

start = time.time()

for idx, row in df.iterrows():
    row_number = idx + 1
    if not tracker.should_process(row_number):
        continue

    text = str(row[TEXT_COLUMN])
    print(f'\n🔍 Row {row_number}/{len(df)}  ({len(text)} chars)')

    result = classifier.process_row(text, row_number)

    if result['stage'] == 'filtered':
        print(f'   ⏩ FILTERED — {result["reason"]}')
    elif result['stage'] == 'not_applicable':
        print(f'   🔇 NOT APPLICABLE (1_mar_prac=4)')
    elif result['stage'] == 'classified':
        c     = result['classification']
        flags = c.get('12_validation_flags') or []
        flag_str = f'  ⚑ REVIEW: {flags}' if flags else ''
        print(
            f'   ✓  subj={c.get("0_subject")}  '
            f'mar={c.get("1_mar_prac")}  '
            f'pol={c.get("3_pol_struc")}  '
            f'moral={c.get("9_moral_op")}  '
            f'conf={c.get("11_code_conf")}'
            f'{flag_str}'
        )
        if c.get('11_conf_note'):
            print(f'   📝 {c["11_conf_note"]}')
    else:
        print(f'   ❌ ERROR — {result.get("error")}')

    tracker.save_row(row_number, result)

    if row_number < len(df):
        time.sleep(REQUEST_DELAY)

    if row_number % BATCH_SIZE == 0:
        elapsed   = time.time() - start
        rate      = row_number / elapsed
        remaining = (len(df) - row_number) / rate if rate > 0 else 0
        print(f'\n   📊 {row_number}/{len(df)} ({row_number/len(df)*100:.1f}%) '
              f'| {rate:.2f} rows/sec | ~{remaining/60:.1f} min remaining')

classifier.print_stats()

print('\n📥 Consolidating results (already on Drive — assembling final JSON)...')
print('   (Results are already on Drive — this step assembles the final JSON.)')
all_results = tracker.load_all_results()
output_path = tracker.finalize(all_results)

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — FLATTEN, SUMMARISE & EXPORT CSVs  (Codebook v6)
# ════════════════════════════════════════════════════════

# Valid codes per variable (numeric coded vars only)
VALID_CODES = {
    '0_subject':      {1: 'african_african', 2: 'african_african_diff',
                       3: 'african_foreign', 4: 'foreign_foreign'},
    '1_mar_prac':     {1: 'monogamy_only', 2: 'polygamy_only',
                       3: 'both', 4: 'not_applicable'},
    '2_mar_diff':     {1: 'widespread', 2: 'rare_exception'},
    '3_pol_struc':    {1: 'fixed_spouses', 2: 'varied', 3: 'unlimited'},
    '5_soc_stat':     {1: 'general_pop', 2: 'elite_wealthy', 3: 'eth_restric'},
    '6_obs_type':     {1: 'auth_direct', 2: 'auth_second_res',
                       3: 'auth_hearsay', 4: 'auth_gen'},
    '7_obs_location': {1: 'yes', 2: 'no'},
    '9_moral_op':     {1: 'moral_judge', 2: 'no_judge'},
    '11_code_conf':   {1: 'high', 2: 'medium', 3: 'low'},
}

# String / open-ended variables (v6 additions included)
STRING_VARS = [
    '4_ethnic_group', '8_cited_loc', '10_stance', '11_conf_note',
    '13_wives_no_notes', '14_concubines', '14_concubines_notes',
    '15_dowry', '15_dowry_notes', '16_gender_roles',
    '17_legality', '18_kingdoms', '19_geo_features', '20_historian_flag',
]

# Flatten all coded rows
rows = []
for r in all_results:
    if r.get('stage') in ('classified', 'not_applicable') and r.get('classification'):
        entry = {'row_number': r['row_number'], 'stage': r['stage']}
        entry.update(r['classification'])
        flags = entry.get('12_validation_flags')
        entry['12_validation_flags'] = '|'.join(flags) if isinstance(flags, list) else ''
        # Convert list fields to pipe-delimited strings for CSV
        for sv in ['4_ethnic_group', '8_cited_loc', '18_kingdoms', '19_geo_features']:
            val = entry.get(sv)
            if isinstance(val, list):
                entry[sv] = ' | '.join(val)
        rows.append(entry)

df_coded    = pd.DataFrame(rows)
df_filtered = pd.DataFrame([
    {'row_number': r['row_number'], 'reason': r.get('reason', '')}
    for r in all_results if r.get('stage') == 'filtered'
])
df_flagged  = df_coded[
    df_coded['12_validation_flags'].str.len() > 0
].copy() if '12_validation_flags' in df_coded.columns else pd.DataFrame()

# ── Out-of-codebook code check ───────────────────────────────────────────
print('\n── Out-of-codebook code check ──────────────────────────')
for var, valid in VALID_CODES.items():
    if var not in df_coded.columns:
        continue
    col = pd.to_numeric(df_coded[var], errors='coerce')
    bad = col.dropna()[~col.dropna().isin(valid.keys())]
    if len(bad) > 0:
        print(f'  ⚠️  {var}: unexpected codes {bad.unique().tolist()} ({len(bad)} rows)')
    else:
        print(f'  ✓  {var}')

# ── v6 new variable fill rates ───────────────────────────────────────────
print('\n── v6 new variable fill rates ──────────────────────────')
v6_vars = {
    '13_wives_no':       'wife count stated',
    '14_concubines':     'concubines mentioned',
    '15_dowry':          'dowry/bride-wealth mentioned',
    '16_gender_roles':   'extreme gender subservience described',
    '17_legality':       'law or rule stated',
    '18_kingdoms':       'African kingdoms/chiefdoms named',
    '19_geo_features':   'geographic features named',
    '20_historian_flag': 'flagged as historian case study',
}
for var, desc in v6_vars.items():
    if var not in df_coded.columns:
        print(f'  {var:<22} — column missing')
        continue
    n_total  = len(df_coded)
    n_filled = df_coded[var].notna().sum()
    pct = n_filled / n_total * 100 if n_total else 0
    print(f'  {var:<22} {n_filled:>5} / {n_total} rows filled ({pct:.1f}%) — {desc}')

# ── Frequency tables for coded variables ────────────────────────────────
print(f'\nCoded rows: {len(df_coded)}  |  Filtered: {len(df_filtered)}  |  Flagged: {len(df_flagged)}')
for var, codes in VALID_CODES.items():
    if var not in df_coded.columns:
        continue
    print(f'\n── {var} ' + '─' * max(1, 42 - len(var)))
    vc = df_coded[var].value_counts(dropna=False).sort_index()
    for code, count in vc.items():
        pct   = count / len(df_coded) * 100
        label = codes.get(code, 'null' if pd.isna(code) else f'UNEXPECTED:{code}')
        marker = ' ⚠️' if isinstance(label, str) and 'UNEXPECTED' in label else ''
        print(f'  {str(code):<4} {label:<22} {count:>5}  ({pct:.1f}%){marker}')

# ── Most flagged variables ───────────────────────────────────────────────
if len(df_flagged) > 0:
    all_flags = []
    for s in df_flagged['12_validation_flags']:
        if s: all_flags.extend(s.split('|'))
    print(f'\n── Most flagged variables ──────────────────────────')
    print(pd.Series(all_flags).value_counts().to_string())

# ── Stage 1 filter reasons ───────────────────────────────────────────────
if len(df_filtered) > 0:
    print(f'\n── Stage 1 filter reasons ──────────────────────────')
    print(df_filtered['reason'].value_counts().to_string())

# ── Historian flags summary ──────────────────────────────────────────────
if '20_historian_flag' in df_coded.columns:
    hist_flagged = df_coded[df_coded['20_historian_flag'].notna()]
    if len(hist_flagged) > 0:
        print(f'\n── 🏛️  Historian case-study flags ({len(hist_flagged)} rows) ──')
        for _, hrow in hist_flagged.iterrows():
            print(f'  Row {int(hrow["row_number"])}: {str(hrow["20_historian_flag"])[:120]}')

# ── Export ───────────────────────────────────────────────────────────────
out = Path(OUTPUT_DIR)
classified_csv = out / 'haiku_v6_coded.csv'
filtered_csv   = out / 'haiku_v6_filtered.csv'
flagged_csv    = out / 'haiku_v6_flagged_for_review.csv'
historian_csv  = out / 'haiku_v6_historian_flags.csv'

df_coded.to_csv(classified_csv, index=False)
df_filtered.to_csv(filtered_csv, index=False)
if len(df_flagged) > 0:
    df_flagged.to_csv(flagged_csv, index=False)
if '20_historian_flag' in df_coded.columns:
    hist_out = df_coded[df_coded['20_historian_flag'].notna()]
    if len(hist_out) > 0:
        hist_out.to_csv(historian_csv, index=False)
        print(f'\n✅ Historian flags CSV     : {historian_csv}')

print(f'\n✅ Coded CSV             : {classified_csv}')
print(f'✅ Filtered CSV          : {filtered_csv}')
print(f'✅ Flagged-for-review CSV: {flagged_csv}')
print(f'\n→ Run Cell 8 to review flagged rows interactively.')


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — HUMAN-IN-THE-LOOP REVIEW  (Codebook v6)
#
# Numeric coded variables: reviewer can accept or override via dropdown.
# v6 string variables: shown read-only for context; edit in CSV if needed.
# Saves: haiku_v6_reconciled.csv
# ════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, clear_output

VALID_CODES = {
    '0_subject':      {1: 'african_african', 2: 'african_african_diff',
                       3: 'african_foreign', 4: 'foreign_foreign'},
    '1_mar_prac':     {1: 'monogamy_only', 2: 'polygamy_only',
                       3: 'both', 4: 'not_applicable'},
    '2_mar_diff':     {1: 'widespread', 2: 'rare_exception'},
    '3_pol_struc':    {1: 'fixed_spouses', 2: 'varied', 3: 'unlimited'},
    '5_soc_stat':     {1: 'general_pop', 2: 'elite_wealthy', 3: 'eth_restric'},
    '6_obs_type':     {1: 'auth_direct', 2: 'auth_second_res',
                       3: 'auth_hearsay', 4: 'auth_gen'},
    '7_obs_location': {1: 'yes', 2: 'no'},
    '9_moral_op':     {1: 'moral_judge', 2: 'no_judge'},
}

V6_STRING_VARS = [
    ('13_wives_no',         'Wife count'),
    ('13_wives_no_notes',   'Wife count notes'),
    ('14_concubines',       'Concubines'),
    ('14_concubines_notes', 'Concubines notes'),
    ('15_dowry',            'Dowry/bride-wealth'),
    ('15_dowry_notes',      'Dowry notes'),
    ('16_gender_roles',     'Gender roles (extreme subservience)'),
    ('17_legality',         'Stated law/rule'),
    ('18_kingdoms',         'African kingdoms/chiefdoms'),
    ('19_geo_features',     'Geographic features'),
    ('20_historian_flag',   '🏛️ Historian flag'),
]

ALL_CODED_VARS = list(VALID_CODES.keys())

# ── Load data ─────────────────────────────────────────────────────────────
flagged_csv    = Path(OUTPUT_DIR) / 'haiku_v6_flagged_for_review.csv'
reconciled_csv = Path(OUTPUT_DIR) / 'haiku_v6_reconciled.csv'

if not flagged_csv.exists():
    print('No flagged rows file found. Run Cells 6–7 first.')
else:
    df_review = pd.read_csv(flagged_csv)

    df_texts = pd.read_excel(INPUT_PATH)[[TEXT_COLUMN]].reset_index()
    df_texts.columns = ['row_index', TEXT_COLUMN]
    df_texts['row_number'] = df_texts['row_index'] + 1
    df_review = df_review.merge(df_texts[['row_number', TEXT_COLUMN]],
                                on='row_number', how='left')

    for var in ALL_CODED_VARS:
        if var in df_review.columns:
            df_review[f'{var}_llm']   = df_review[var]
            df_review[f'{var}_final'] = df_review[var]

    df_review['review_status'] = 'pending'
    df_review['reviewer_note'] = ''

    state = {'idx': 0, 'total': len(df_review), 'decisions': {}}
    out_display = widgets.Output()

    def render_row(idx):
        out_display.clear_output(wait=True)

        if idx >= state['total']:
            with out_display:
                print('\n🎉 All flagged rows reviewed!')
                save_reconciled()
            return

        row = df_review.iloc[idx]
        flags_str    = str(row.get('12_validation_flags', ''))
        flagged_vars = [v.strip() for v in flags_str.split('|')
                        if v.strip() in VALID_CODES]

        row_num   = int(row['row_number'])
        text_full = str(row.get(TEXT_COLUMN, ''))
        text_show = text_full[:600] + ('…' if len(text_full) > 600 else '')

        with out_display:
            print(f'\n{"─"*65}')
            print(f'ROW {row_num}  [{idx+1} of {state["total"]}]  '
                  f'1_mar_prac={row.get("1_mar_prac")}  conf={row.get("11_code_conf")}')
            if row.get('11_conf_note') and str(row['11_conf_note']) != 'nan':
                print(f'LLM note: {row["11_conf_note"]}')
            hist_flag = row.get('20_historian_flag')
            if hist_flag and str(hist_flag) != 'nan':
                print(f'🏛️  HISTORIAN FLAG: {hist_flag}')
            print(f'{"─"*65}')
            print(f'TEXT:\n{text_show}')
            print(f'{"─"*65}')
            print(f'FLAGGED VARIABLES: {flagged_vars}')
            print()

            print('  ── Numeric codes ──')
            for var in ALL_CODED_VARS:
                if var in df_review.columns:
                    code  = row.get(var)
                    label = VALID_CODES[var].get(code, str(code)) if pd.notna(code) else 'null'
                    flag_marker = ' ⚑' if var in flagged_vars else ''
                    print(f'  {var:<20} {str(code) if pd.notna(code) else "null":<4} {label}{flag_marker}')

            print('\n  ── v6 string variables ──')
            for var, label in V6_STRING_VARS:
                val = row.get(var)
                if pd.notna(val) and str(val) not in ('nan', ''):
                    print(f'  {label:<36} {str(val)[:100]}')

        dropdowns = {}
        for var in flagged_vars:
            codes = VALID_CODES[var]
            current_code = row.get(var)
            options = [(f'{k} — {v}', k) for k, v in codes.items()]
            current_val = int(current_code) if pd.notna(current_code) else list(codes.keys())[0]
            dd = widgets.Dropdown(
                options=options, value=current_val,
                description=f'{var}:',
                style={'description_width': '160px'},
                layout=widgets.Layout(width='480px')
            )
            dropdowns[var] = dd

        note_box = widgets.Text(
            value='', placeholder='Optional reviewer note',
            description='Note:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='560px')
        )
        btn_save = widgets.Button(description='Save & Next ▶', button_style='success',
                                   layout=widgets.Layout(width='160px'))
        btn_skip = widgets.Button(description='Skip (keep LLM codes)', button_style='warning',
                                   layout=widgets.Layout(width='200px'))
        progress_bar = widgets.IntProgress(
            value=idx, min=0, max=state['total'],
            description=f'{idx}/{state["total"]}', bar_style='info',
            layout=widgets.Layout(width='400px')
        )

        def on_save(_):
            decisions = {var: dd.value for var, dd in dropdowns.items()}
            state['decisions'][row_num] = decisions
            for var, final_code in decisions.items():
                df_review.at[idx, f'{var}_final'] = final_code
            df_review.at[idx, 'review_status'] = 'reviewed'
            df_review.at[idx, 'reviewer_note'] = note_box.value
            state['idx'] += 1
            render_row(state['idx'])

        def on_skip(_):
            df_review.at[idx, 'review_status'] = 'skipped'
            state['idx'] += 1
            render_row(state['idx'])

        btn_save.on_click(on_save)
        btn_skip.on_click(on_skip)
        display(widgets.VBox(list(dropdowns.values()) + [note_box,
                widgets.HBox([btn_save, btn_skip]), progress_bar]))

    def save_reconciled():
        out_cols = ['row_number', 'stage', 'review_status', 'reviewer_note',
                    '1_mar_prac', '11_code_conf', '11_conf_note',
                    '12_validation_flags', TEXT_COLUMN]
        for var in ALL_CODED_VARS:
            if f'{var}_llm' in df_review.columns:
                out_cols += [f'{var}_llm', f'{var}_final']
        for var, _ in V6_STRING_VARS:
            if var in df_review.columns:
                out_cols.append(var)
        for sv in ['4_ethnic_group', '8_cited_loc', '10_stance']:
            if sv in df_review.columns:
                out_cols.append(sv)
        out_cols = [c for c in dict.fromkeys(out_cols) if c in df_review.columns]
        df_review[out_cols].to_csv(reconciled_csv, index=False)
        with out_display:
            reviewed = (df_review['review_status'] == 'reviewed').sum()
            skipped  = (df_review['review_status'] == 'skipped').sum()
            pending  = (df_review['review_status'] == 'pending').sum()
            print(f'\n✅ Reconciled CSV saved: {reconciled_csv}')
            print(f'   Reviewed: {reviewed}  Skipped: {skipped}  Pending: {pending}')
            print('   v6 string fields are in the CSV as-is — edit directly in Sheets if needed.')

    btn_save_now = widgets.Button(description='💾 Save progress now', button_style='info',
                                   layout=widgets.Layout(width='200px'))
    btn_save_now.on_click(lambda _: save_reconciled())

    if state['total'] == 0:
        print('✓ No flagged rows to review — all LLM codes are high-confidence.')
    else:
        print(f'Starting review of {state["total"]} flagged rows (Codebook v6).')
        print('Numeric flags: adjust via dropdown. v6 strings: shown for context; edit CSV if needed.')
        print('Click "💾 Save progress now" at any time.\n')
        display(btn_save_now)
        display(out_display)
        render_row(0)
